# GLM Algorithm

In [ ]:
#
# The Cox proportional hazards model is a semi-parametric regression technique used 
# for survival analysis to examine the relationship between covariates and the time 
# until an event of interest occurs (like disease progression or death). I suggest 
# to have a brief look at the swimlane diagram in the security and privacy section 
# (../security-and-privacy/Security & Privacy CoxPH.pdf) to have a good 
# overview of the different steps in the algorithm. From the diagram you can see that 
# this is a three step (federated-)step itterative algorithm:
#
# 1. Call `extract_distinct_event_times`
# 2. Call `compute_summed_z`
# 3. Call `compute_relevant_matrices`
#
# These two steps are repeated until the algorithm converges. And then there is the 
# central part responsible for the aggregation of the results. The central part of the 
# algorithm (the main call) will return the GLM estimate for the entire federated dataset. 
#
# 1. Create a new vantage6 task to execute the *glm* method (central 
#    part). This central part will start the tasks `compute_local_betas` and
#    `compute_local_deviance` (as you can see in the swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* GLM estimate from the central part (the main call)
#

In [ ]:
import base64
import json
import requests

In [ ]:
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {
    "Authorization": token
}

In [ ]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 3

In [ ]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1	- root (UPM)
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#   8	- VGR
#   9   - OUS
#   10  - MSCI
#   11  - APHP
#
# These are basically all organization that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATION_IDS = [org["id"] for org in response.json()["data"]]
ORGANIZATION_IDS

In [ ]:
ORGANIZATION_IDS = [4]

In [ ]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 253
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 222

In [ ]:
# The image to use is the latest version of the analytics algorithm
IMAGE = "ghcr.io/iknl/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "coxph_central"

In [ ]:
# The user is able to select multiple cohorts in RAVEN. These cohorts correspond to
# different dataframes, see the `2-new-cohort.ipynb` how these are created and check
# the `v6_dataframe` column in the RAVEN database for more information.
#
# In this example I've just used one cohort. Simply add more ids to the list to use
# more cohorts.
DATAFRAME_IDS = [513]

In [ ]:
payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": [
        {
            "id": ORGANIZATION_IDS[0], # Central task
            "arguments": base64.b64encode(
                json.dumps(
                    {
                        "outcome_col": "censor",
                        "expl_vars": ["age", "year_of_birth"],
                        "time_col": "new_surv",
                        "organizations_to_include": ORGANIZATION_IDS,
                    }
                ).encode("UTF-8")
            ).decode("UTF-8")
        }
    ],
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}
payload

In [ ]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

In [ ]:
# Poll until the (central) task is finished. We do not concern about the subtasks in
# this instance. We could consider including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

In [ ]:
# Get the results of the (central) task, thus the *global* summary statistics.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)
# Again, since it is a central task we can obtain the [0]th element of the data list
json.loads(base64.b64decode(response.json()["data"][0]["result"]).decode("UTF-8"))

result_global = json.loads(base64.b64decode(response.json()["data"][0]["result"]).decode("UTF-8"))

In [ ]:
result_global

In [ ]:
#
# Sometimes you do net get a result.
#
# Model did not converge: 
# {'cohorts': {}, 'details': {'iterations': 25, 'all_converged': False}}
#
# Singulairity issue:
# {'cohorts': {'cox2_f88083a9': {'msg': "Newton step failed for cohort 'cox2_f88083a9': cannot solve the Fisher information system (singular matrix). Likely causes: perfect multicollinearity (e.g. age and year_of_birth are linearly dependent), complete separation, or numerical overflow from diverging betas. Consider removing collinear or redundant predictors."}},
# 'details': {'iterations': 2, 'all_converged': True}}
#
#
# A valid result looks like this:
# {'cohorts': {'cox2_f88083a9': {'included_organizations': [4],
#    'excluded_organizations': [],
#    'model': '{"Coef":{"age":-0.01732,"sex[T.MALE]":0.07826,"sex[T.No matching concept]":2.29277},"Exp(coef)":{"age":0.98283,"sex[T.MALE]":1.08141,"sex[T.No matching concept]":9.90232},"SE":{"age":0.00289,"sex[T.MALE]":0.10886,"sex[T.No matching concept]":1.01301},"lower_CI":{"age":0.97728,"sex[T.MALE]":0.87362,"sex[T.No matching concept]":1.35971},"upper_CI":{"age":0.98841,"sex[T.MALE]":1.3386,"sex[T.No matching concept]":72.11554},"Z":{"age":-5.9444290759,"sex[T.MALE]":0.7478399955,"sex[T.No matching concept]":8.7880169797},"p-value":{"age":0.0000000028,"sex[T.MALE]":0.4545566745,"sex[T.No matching concept]":1.522229681e-18}}',
#    'overall_p_value': 9.102483675309797e-09,
#    'aic': 4054.7914895952936,
#    'degrees_of_freedom': 3,
#    'warnings': [],
#    'converged': True}},
#  'details': {'iterations': 7, 'all_converged': True}}